# 08 — Advanced Governance Features

This notebook demonstrates three advanced Data Mesh governance capabilities:

1. **Change Data Feed (CDF)** — Consumers read only incremental changes, not full table scans
2. **Row-Level Security** — Fine-grained access: column masking + row filters
3. **Automated Alerting** — SQL Alert fires when data contracts are violated

These features turn passive governance (documentation) into **active governance** (enforcement).

## 1. Change Data Feed (CDF)

CDF is enabled on all 3 gold tables. Consumers can now:
- Read only rows that changed since their last checkpoint
- See `_change_type` (insert, update_preimage, update_postimage, delete)
- Build efficient incremental pipelines without full scans

This is critical for Data Mesh: consumers shouldn't need to re-process all data every time the product refreshes.

In [0]:
%sql
-- Tag gold DATA PRODUCT tables with mesh metadata
ALTER TABLE adoit_product.gold.application_landscape           SET TAGS ('data_product' = 'true', 'domain' = 'EA', 'pii' = 'false', 'classification' = 'internal');
ALTER TABLE snow_product.gold.service_health                    SET TAGS ('data_product' = 'true', 'domain' = 'ITSM', 'pii' = 'false', 'classification' = 'internal');
ALTER TABLE data_mesh_hub.cross_domain.application_risk_profile SET TAGS ('data_product' = 'true', 'domain' = 'Cross-Domain', 'pii' = 'false', 'classification' = 'confidential');

-- Enable Change Data Feed on gold tables for incremental consumers
ALTER TABLE adoit_product.gold.application_landscape SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE snow_product.gold.service_health SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
ALTER TABLE data_mesh_hub.cross_domain.application_risk_profile SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
# Verify CDF is enabled on gold tables
tables = [
    "adoit_product.gold.application_landscape",
    "snow_product.gold.service_health",
    "data_mesh_hub.cross_domain.application_risk_profile"
]
for t in tables:
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {t}").filter("key = 'delta.enableChangeDataFeed'").collect()
        enabled = props[0]['value'] if props else 'not set'
        print(f"✓ {t}: CDF = {enabled}")
    except Exception as e:
        print(f"⚠ {t}: {e}")

In [0]:
# Read CDF changes — may be empty if no changes occurred after CDF was enabled
try:
    df = spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", 0).table("adoit_product.gold.application_landscape")
    display(df.select("app_name", "lifecycle_status", "_change_type", "_commit_version", "_commit_timestamp"))
except Exception as e:
    print(f"CDF not yet available (expected on fresh rebuild): {type(e).__name__}")
    print("Changes will be captured from the next MERGE operation onwards.")

In [0]:
%sql
-- Consumers can also use timestamps instead of versions:
-- SELECT * FROM table_changes('adoit_product.gold.application_landscape', '2026-04-17T00:00:00')
-- Or in Spark Structured Streaming:
-- spark.readStream.format('delta').option('readChangeFeed', 'true').option('startingVersion', 4).table('adoit_product.gold.application_landscape')

## 2. Row-Level Security

### Column Masking
Sensitive contact fields are masked for non-privileged users:
- `data_product_registry.owner_contact` → `ea-***@***.com`
- `data_contracts.escalation_contact` → `its***@***.com`

### Row Filtering
The `risk-committee` group can only see HIGH and CRITICAL risk applications in `application_risk_profile`. Other groups see all rows.

This implements **Attribute-Based Access Control (ABAC)** — going beyond table-level GRANTs to field-level and row-level restrictions.

In [0]:
%sql
-- Create column mask function for PII protection
CREATE OR REPLACE FUNCTION data_mesh_hub.platform.mask_contact(contact STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('admins') THEN contact
  ELSE REGEXP_REPLACE(contact, '(.)(.*)(@.*)', '\1***\3')
END;

-- View the mask function
DESCRIBE FUNCTION data_mesh_hub.platform.mask_contact;

In [0]:
%sql
-- As admin, we see full data:
SELECT product_name, owner_contact FROM data_mesh_hub.platform.data_product_registry;
-- A non-admin user would see: 'ea-***@***.com' instead of 'ea-team@company.com'

In [0]:
%sql
-- Create row filter function for risk-based access control
CREATE OR REPLACE FUNCTION data_mesh_hub.platform.filter_by_risk(risk_level STRING)
RETURNS BOOLEAN
RETURN CASE
  WHEN is_account_group_member('risk-committee') THEN TRUE
  WHEN risk_level IN ('HIGH', 'CRITICAL') THEN FALSE
  ELSE TRUE
END;

-- View the filter function
DESCRIBE FUNCTION data_mesh_hub.platform.filter_by_risk;

## 3. Automated Contract Violation Alert

A **Databricks SQL Alert** monitors all active data contracts. When any contract's SLA or quality terms are violated:
1. The alert query returns the violated contracts
2. An alert fires with the subject: *"⚠️ Data Contract Violation Detected"*
3. The escalation contact and channel from the contract are included

This turns contracts from **passive documentation** into **active enforcement**.

**Alert ID:** `8548d7a9-16d2-4a3b-b281-d037eb053798`  
**Query ID:** `5c630b22-2ddc-4e07-bab5-1aa4dd999636`

In [0]:
%sql
-- The alert query: returns rows ONLY when violations exist
SELECT 
  c.contract_id, c.product_name, c.consumer_group, c.escalation_contact,
  CASE WHEN o.freshness_hours > c.agreed_sla_hours THEN 'SLA_VIOLATED' ELSE 'OK' END AS sla_check,
  CASE WHEN o.quality_score < c.quality_threshold_pct THEN 'QUALITY_VIOLATED' ELSE 'OK' END AS quality_check,
  ROUND(o.freshness_hours, 1) AS actual_freshness_hrs, c.agreed_sla_hours,
  ROUND(o.quality_score, 1) AS actual_quality_pct, c.quality_threshold_pct
FROM data_mesh_hub.platform.data_contracts c
JOIN (
  SELECT product_id, quality_score, freshness_hours,
         ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY check_timestamp DESC) AS rn
  FROM data_mesh_hub.platform.product_observability
) o ON o.product_id = c.product_id AND o.rn = 1
WHERE c.contract_status = 'active'
  AND (o.freshness_hours > c.agreed_sla_hours OR o.quality_score < c.quality_threshold_pct)

## Summary: Active vs Passive Governance

| Feature | Before (Passive) | After (Active) |
|---|---|---|
| **Data Changes** | Consumer scans full table every run | CDF: consumer reads only incremental changes |
| **Access Control** | Table-level GRANT/REVOKE | Column masking + row filters (ABAC) |
| **Contract Enforcement** | Manual review of compliance table | SQL Alert fires automatically on violations |
| **Lineage** | TBLPROPERTIES metadata | Sankey diagram on dashboard + UC automatic lineage |

These features demonstrate that Databricks Unity Catalog provides **production-grade governance** for a Data Mesh — not just documentation, but enforcement.